# 08N: Object registry and reusable patch preparation

Reuse the fixed model revision and saved masks. Preserve all valid object candidates and prepare selected-view object banks. Neutral correspondence is an explicitly assisted route. Anomaly labels remain in the evaluator.

## Saved-output evidence copy

This notebook records the completed run for repository review. Its code and markdown match the clean runnable notebook. Private scene visuals and identifier-bearing output were removed. Safe aggregate tables and charts were retained.

Run stage: `08N local development`  
Clean notebook SHA-256: `c71d1d5212287edfc55d7e467e424729305c9d10d591caf466fd1665ae5f2622`  
Downloaded saved-output SHA-256: `9dc30131fb24be0f4196c180169ecf3aa2ee9bb9fd511848555aa50a4992933a`


## Frozen comparison

LocateAnything-3B, SAM 2.1 and DINOv2-small remain fixed. The existing 100-scene development screen supplies 19 predefined schedules: five each with 1, 2 and 3 views, and four with 5 views. Three and five views are the application settings; one and two views are diagnostics. Every prediction uses only its selected views. This bounded stage does not claim results for 10 or 20 views.

| Route | Object regions | Cross-view identity | Interpretation |
| --- | --- | --- | --- |
| `supplied_reference` | Supplied masks | Supplied neutral correspondence | Scoring diagnostic |
| `automatic_reference` | Saved LocateAnything and SAM outputs | Supplied neutral correspondence | Region diagnostic; assisted identity |
| `automatic_predicted` | Saved LocateAnything and SAM outputs | Existing automatic association | Automatic application route |

Anomaly labels are available only to the evaluator, after predictions are sealed. All valid candidates remain represented. An unmatched observation stays visible in the output; its object-bank score is unavailable because exclusion of another view of itself cannot be guaranteed. Missing observations are never replaced with normal scores. One object cannot contain two detections from the same view.

**Patch comparison.** For each target foreground patch, compute the minimum cosine distance to a peer object's patches across that peer's selected views. Each peer contributes one distance. Take the lower median across peers, then average the largest `max(1, floor(0.02 × target patch count))` patch distances. Exclude the target's complete identity from all peer banks. At least two usable peers are required.

Compare this object bank with regenerated within-view scoring and cached forward scoring. `view_mean_control` first averages nearest-patch distances across each peer's usable views. It diagnoses the extra matching opportunities in a larger bank; it is not a guarantee of invariance to bank size.

**One physical-object ranking.** Convert available observation scores into average within-view percentile ranks when at least three candidates are scored. Compare the mean of all valid ranks with the mean of the strongest two (or the single available rank). Retain the existing winner-consensus method as a comparison. The scorer's raw-score means are diagnostics; final rankings use the declared rank normalization.

**Decision gate.** Choose between the two bank fusions using the supplied-reference primary three-view development result: the paired asset-bootstrap 95% lower bound for top-1 improvement must be positive, AP and Recall@2 must not regress, and the mean top-1 improvement across the five three-view schedules must be positive. Prefer mean fusion on a tie. If the gate fails, stop the scoring extension and retain the stronger existing method. Results remain development evidence; 60% is a goal, not a promised outcome.

Report tie-aware top-1, Recall@2, reciprocal rank and average precision, with asset-grouped uncertainty and explicit candidate, association and ranking losses. A scene contributes one result per setting; its views are not independent successes.

## Runtime and configuration

In [1]:
from pathlib import Path
import json
import os
import sys

def is_source(path):
    return (path / "scripts/run_object_bank.py").is_file() and (path / "src/vpi_object_bank").is_dir()

def locate_source():
    explicit = os.environ.get("VPI_OBJECT_BANK_ROOT")
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not is_source(root):
            raise RuntimeError("The configured source root is incomplete")
        return root
    for root in (Path.cwd(), *list(Path.cwd().parents)[:6]):
        if is_source(root):
            return root
    candidates = set()
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for pattern in ("*", "datasets/*/*", "datasets/*/*/versions/*", "notebooks/*/*"):
            for mount in input_root.glob(pattern):
                if not mount.is_dir():
                    continue
                if is_source(mount):
                    candidates.add(mount.resolve())
                children = list(mount.iterdir())
                if len(children) <= 200:
                    candidates.update(child.resolve() for child in children if child.is_dir() and is_source(child))
    if len(candidates) != 1:
        raise RuntimeError("Expected exactly one complete object-bank source resource")
    return candidates.pop()

SOURCE_ROOT = locate_source()
CONFIG_PATH = Path(os.environ.get("VPI_OBJECT_BANK_CONFIG", SOURCE_ROOT / ".local/config.json")).expanduser().resolve()
if not CONFIG_PATH.is_file():
    raise RuntimeError("Provide the verified local or Kaggle configuration JSON")
configuration = json.loads(CONFIG_PATH.read_text())
sys.path.insert(0, str(SOURCE_ROOT / "scripts"))
sys.path.insert(0, str(SOURCE_ROOT / "src"))
from scoring_comparison_runtime import run_bounded
os.environ["PYTHONPATH"] = str(SOURCE_ROOT / "src")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
LOG_ROOT = SOURCE_ROOT / ".local/logs"
if str(SOURCE_ROOT).startswith("/kaggle/input/"):
    LOG_ROOT = Path("/kaggle/working/object-bank-logs")
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    run_bounded(
        [sys.executable, str(SOURCE_ROOT / "scripts/run_object_bank.py"), stage, "--config", str(CONFIG_PATH)],
        LOG_ROOT / (stage + ".log"), idle_seconds=900, total_seconds=18000,
    )
print({"event": "notebook_ready", "python": sys.version.split()[0]})

{'event': 'notebook_ready', 'python': '3.12.14'}


## Input preflight

In [2]:
run_stage("preflight")

{"stage": "preflight", "scenes": 100, "protocol_sha256": "52011f63a676a5abf549b884deb3e76f5603a92e53aa9e0147ce80a6378f2fc3", "status": "verified"}


## Reusable patch extraction

In [3]:
run_stage("extract")

{"result": {"completed_units": 4000, "elapsed_seconds": 0.0, "extracted_units": 0, "sealed": true, "total_units": 4000}, "stage": "extract"}


## Object registries and predictions

In [4]:
run_stage("predict")

{"completed":1,"computed":1,"elapsed_seconds":0.5985918330006825,"event":"progress","reused":0,"total":100}


{"completed":2,"computed":2,"elapsed_seconds":1.3461859160015592,"event":"progress","reused":0,"total":100}


{"completed":3,"computed":3,"elapsed_seconds":1.9603225829996518,"event":"progress","reused":0,"total":100}


{"completed":4,"computed":4,"elapsed_seconds":2.533294000000751,"event":"progress","reused":0,"total":100}


{"completed":5,"computed":5,"elapsed_seconds":3.033793582999351,"event":"progress","reused":0,"total":100}


...

{"completed":97,"computed":97,"elapsed_seconds":50.30132175000108,"event":"progress","reused":0,"total":100}


{"completed":98,"computed":98,"elapsed_seconds":50.71640070799913,"event":"progress","reused":0,"total":100}


{"completed":99,"computed":99,"elapsed_seconds":51.06700929100043,"event":"progress","reused":0,"total":100}


{"completed":100,"computed":100,"elapsed_seconds":51.63526895800169,"event":"progress","reused":0,"total":100}


{"result": {"computed": 100, "elapsed_seconds": 51.72542387499925, "reused": 0, "scenes": 100, "sealed": true}, "stage": "predict"}


## Verify sealed predictions

In [5]:
run_stage("verify")

{"result": {"feature_root": {"verified_files": 12002}, "prediction_root": {"verified_files": 301}, "prepared_root": {"verified_files": 101}}, "stage": "verify"}


## Protocol and reusable cache evidence

In [6]:
from IPython.display import display
import pandas as pd
from vpi_object_bank.protocol import PROTOCOL

# The protocol has no evaluator labels or machine-specific paths.
display(PROTOCOL)
schedule_rows = [{"schedule": key, "view_count": len(views), "views": ", ".join(views)}
                 for key, views in PROTOCOL["view_sets"].items()]
display(pd.DataFrame(schedule_rows))
feature_root = Path(configuration["feature_root"])
feature_lock = json.loads((feature_root / "feature_lock.json").read_text())
binding = json.loads((feature_root / "run_binding.json").read_text())
progress = json.loads((feature_root / "progress.json").read_text())
metadata = feature_lock.get("metadata", {})
display({"sealed_scene_count": len(metadata.get("scenes", [])),
         "cached_view_count": len(metadata.get("views", [])),
         "completed_scene_region_view_units": metadata.get("completed_units"),
         "last_extraction_progress": {key: progress.get(key) for key in
              ("completed", "total", "extracted", "reused", "elapsed_seconds")},
         "device": binding.get("device"), "runtime": binding.get("runtime"),
         "model_file_sha256": binding.get("model_files")})
print("The extraction cell reports zero extracted units when an identical sealed cache is reused.")

{'schema': 'vpi-object-bank/v1',
 'scope': 'existing 100-scene development screen; no new roles or protected data',
 'view_sets': {'one-primary': ['V00'],
  'one-rotated-04': ['V04'],
  'one-rotated-08': ['V08'],
  'one-rotated-12': ['V12'],
  'one-rotated-16': ['V16'],
  'two-primary': ['V00', 'V10'],
  'two-rotated-04': ['V04', 'V14'],
  'two-rotated-08': ['V08', 'V18'],
  'two-rotated-02': ['V02', 'V12'],
  'two-rotated-06': ['V06', 'V16'],
  'three-primary': ['V00', 'V07', 'V14'],
  'three-rotated-04': ['V04', 'V11', 'V18'],
  'three-rotated-08': ['V02', 'V08', 'V15'],
  'three-rotated-12': ['V06', 'V12', 'V19'],
  'three-rotated-16': ['V03', 'V10', 'V16'],
  'five-primary': ['V00', 'V04', 'V08', 'V12', 'V16'],
  'five-rotated-01': ['V01', 'V05', 'V09', 'V13', 'V17'],
  'five-rotated-02': ['V02', 'V06', 'V10', 'V14', 'V18'],
  'five-rotated-03': ['V03', 'V07', 'V11', 'V15', 'V19']},
 'extraction': {'encoder': 'facebook/dinov2-small',
  'revision': 'ed25f3a31f01632728cabb09d1542f84a

,schedule,view_count,views
0,one-primary,1,V00
1,one-rotated-04,1,V04
2,one-rotated-08,1,V08
3,one-rotated-12,1,V12
4,one-rotated-16,1,V16
5,two-primary,2,"V00, V10"
6,two-rotated-04,2,"V04, V14"
7,two-rotated-08,2,"V08, V18"
8,two-rotated-02,2,"V02, V12"
9,two-rotated-06,2,"V06, V16"


{'sealed_scene_count': 100,
 'cached_view_count': 20,
 'completed_scene_region_view_units': 4000,
 'last_extraction_progress': {'completed': 4000,
  'total': 4000,
  'extracted': 4000,
  'reused': 0,
  'elapsed_seconds': 365.7540948749993},
 'device': 'cpu',
 'runtime': {'numpy': '2.5.3',
  'pillow': '12.3.0',
  'safetensors': '0.8.0',
  'torch': '2.13.0',
  'transformers': '5.14.1'},
 'model_file_sha256': {'config.json': '1809f83e3bdb1609a501a610ad4a742f4fd8ae44d72ca4aa0df52d1f2ac8628d',
  'model.safetensors': 'ae1e99fcefd534ed978cdeb8326f08030c96e28b7a81ffcbc98a857c84d14be1'}}

The extraction cell reports zero extracted units when an identical sealed cache is reused.
